# YETI Valuation-to-Option Pricing and Greeks Lab
Completed reproducible artifact for Modules 4 and 5. Analyst scenarios are belief-based; CRR is risk-neutral; Greeks are local sensitivities.

## 1. Load and validate inputs

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
from scipy.stats import norm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv'); targets=pd.read_csv(DATA/'analyst_targets.csv'); sources=pd.read_csv(DATA/'source_documentation.csv')
assert len(inputs)==1 and len(targets)>=5
M=inputs.iloc[0].to_dict()
num=['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','contract_multiplier','dcf_value','comps_value','blended_value']
for k in num: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
assert 0<M['volatility']<=3 and 0<=M['risk_free_rate']<=1 and M['market_bid']<=M['market_ask']
M

{'company': 'YETI Holdings Inc.',
 'ticker': 'YETI',
 'valuation_date': '2026-11-13',
 'equity_valuation_date': '2026-09-15',
 'spot': 39.42,
 'strike': 40.0,
 'expiration_date': '2026-11-20',
 'maturity_years': 0.0191780821917808,
 'risk_free_rate': 0.0432,
 'dividend_yield': 0.0,
 'volatility': 0.379,
 'volatility_type': 'expiration-level ATM implied',
 'option_type': 'call',
 'exercise_style': 'american',
 'lattice_steps': 500,
 'market_bid': 0.8,
 'market_ask': 1.2,
 'contract_multiplier': 100.0,
 'volume': 129,
 'open_interest': 80,
 'dcf_value': 37.2974910118166,
 'comps_value': 49.8128904048684,
 'blended_value': 42.3036507690373,
 'market_mid': 1.0}

## 2. Equity valuation and analyst distribution

In [2]:
stats={'count':len(targets),'mean':targets.target.mean(),'median':targets.target.median(),'std':targets.target.std(ddof=1),'min':targets.target.min(),'max':targets.target.max()}
pd.Series({'Current market':M['spot'],'DCF':M['dcf_value'],'Comparable companies':M['comps_value'],'Blended valuation':M['blended_value'],'Analyst mean':stats['mean'],'Analyst median':stats['median']}).to_frame('Value per share')

,Value per share
Current market,39.420000
DCF,37.297491
Comparable companies,49.812890
Blended valuation,42.303651
Analyst mean,51.000000
Analyst median,50.000000


The analyst targets are 12-month valuation opinions. Their horizon materially exceeds the seven-day option maturity, so the scenario value is a belief-based stress comparison rather than an aligned forecast.

## 3. Pricing functions and five states

In [3]:
def payoff(S,K,kind='call'):
    S=np.asarray(S,float); return np.maximum(S-K,0) if kind=='call' else np.maximum(K-S,0)
def crr(S,K,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed')
    j=np.arange(N+1); st=S*u**j*d**(N-j); v=payoff(st,K,kind)
    for n in range(N-1,-1,-1):
        v=disc*(p*v[1:n+2]+(1-p)*v[:n+1])
        if american:
            j=np.arange(n+1); st=S*u**j*d**(n-j); v=np.maximum(v,payoff(st,K,kind))
    return {'price':float(v[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
def bs_price(S,K,T,r,sigma,q=0,kind='call'):
    if T<=0: return float(payoff(S,K,kind))
    d1=(np.log(S/K)+(r-q+sigma*sigma/2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T)
    if kind=='call': return S*np.exp(-q*T)*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
    return K*np.exp(-r*T)*norm.cdf(-d2)-S*np.exp(-q*T)*norm.cdf(-d1)
def bs_greeks(S,K,T,r,sigma,q=0,kind='call'):
    d1=(np.log(S/K)+(r-q+sigma*sigma/2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T); pdf=norm.pdf(d1)
    if kind=='call':
        delta=np.exp(-q*T)*norm.cdf(d1); theta=(-S*np.exp(-q*T)*pdf*sigma/(2*np.sqrt(T))-r*K*np.exp(-r*T)*norm.cdf(d2)+q*S*np.exp(-q*T)*norm.cdf(d1))/365; rho=K*T*np.exp(-r*T)*norm.cdf(d2)*.01
    else:
        delta=np.exp(-q*T)*(norm.cdf(d1)-1); theta=(-S*np.exp(-q*T)*pdf*sigma/(2*np.sqrt(T))+r*K*np.exp(-r*T)*norm.cdf(-d2)-q*S*np.exp(-q*T)*norm.cdf(-d1))/365; rho=-K*T*np.exp(-r*T)*norm.cdf(-d2)*.01
    return {'delta':delta,'gamma':np.exp(-q*T)*pdf/(S*sigma*np.sqrt(T)),'vega_1pct':S*np.exp(-q*T)*pdf*np.sqrt(T)*.01,'theta_day':theta,'rho_1pct':rho}


In [4]:
qmap={'Severe Bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}; weights={'Severe Bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({'state':list(qmap),'quantile':list(qmap.values()),'terminal_stock_price':[targets.target.quantile(q) for q in qmap.values()],'subjective_weight':[weights[x] for x in qmap]})
states['option_payoff']=payoff(states.terminal_stock_price,M['strike'],M['option_type']); states['weighted_payoff']=states.subjective_weight*states.option_payoff
belief_value=np.exp(-M['risk_free_rate']*M['maturity_years'])*states.weighted_payoff.sum(); weight_zero=states.loc[states.option_payoff==0,'subjective_weight'].sum()
print(f'Zero-payoff weight: {weight_zero:.1%}; discounted analyst-scenario value: ${belief_value:.2f}')
states

Zero-payoff weight: 0.0%; discounted analyst-scenario value: $10.72


,state,quantile,terminal_stock_price,subjective_weight,option_payoff,weighted_payoff
0,Severe Bear,0.05,46.6,0.1,6.6,0.66
1,Bear,0.25,48.0,0.2,8.0,1.60
2,Base,0.50,50.0,0.4,10.0,4.00
3,Bull,0.75,54.0,0.2,14.0,2.80
4,Tail,0.95,56.7,0.1,16.7,1.67


## 4. CRR value and required checks

In [5]:
lat=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],True)
euro=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],False)
steps=[25,50,100,200,500]; conv=pd.DataFrame({'steps':steps,'price':[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'],True)['price'] for n in steps]})
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years']); lo=max(pvs-pvk,0); hi=pvs
checks=pd.Series({'0 < p* < 1':0<lat['p_star']<1,'No-arbitrage growth condition':lat['d']<np.exp((M['risk_free_rate']-M['dividend_yield'])*lat['dt'])<lat['u'],'Call bounds':lo<=lat['price']<=hi,'Bid <= ask':M['market_bid']<=M['market_ask'],'Convergence |500-200| < $0.05':abs(conv.price.iloc[-1]-conv.price.iloc[-2])<.05,'American premium negligible':abs(lat['price']-euro['price'])<.01})
print({'CRR':lat['price'],'u':lat['u'],'d':lat['d'],'p*':lat['p_star'],'market midpoint':M['market_mid']}); display(conv); checks.to_frame('pass')

{'CRR': 0.5859654351189759, 'u': 1.0023499931218376, 'd': 0.9976555163985001, 'p*': 0.49976615657944845, 'market midpoint': 1.0}


,steps,price
0,25,0.587706
1,50,0.582459
2,100,0.587809
3,200,0.585283
4,500,0.585965


,pass
0 < p* < 1,True
No-arbitrage growth condition,True
Call bounds,True
Bid <= ask,True
Convergence |500-200| < $0.05,True
American premium negligible,True


## 5. Greeks, conventions, and finite-difference validation
Vega and rho are reported per one-percentage-point change. Theta is per calendar day. Values are per share unless multiplied by the 100-share contract multiplier.

In [6]:
S,K,T,r,q,sig=M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['dividend_yield'],M['volatility']; base=bs_price(S,K,T,r,sig,q,M['option_type']); G=bs_greeks(S,K,T,r,sig,q,M['option_type'])
hS=.01*S; hv=.001; hr=.0001; hT=1/365
fd={'delta':(bs_price(S+hS,K,T,r,sig,q)-bs_price(S-hS,K,T,r,sig,q))/(2*hS),'gamma':(bs_price(S+hS,K,T,r,sig,q)-2*base+bs_price(S-hS,K,T,r,sig,q))/hS**2,'vega_1pct':(bs_price(S,K,T,r,sig+hv,q)-bs_price(S,K,T,r,sig-hv,q))/(2*hv)*.01,'theta_day':bs_price(S,K,T-hT,r,sig,q)-base,'rho_1pct':(bs_price(S,K,T,r+hr,sig,q)-bs_price(S,K,T,r-hr,sig,q))/(2*hr)*.01}
bumps={'delta':hS,'gamma':hS,'vega_1pct':hv,'theta_day':hT,'rho_1pct':hr}
validation=pd.DataFrame([{'Greek':k,'Primary':G[k],'Finite difference':fd[k],'Bump':bumps[k],'Difference':fd[k]-G[k],'Per contract':G[k]*M['contract_multiplier']} for k in G])
validation

,Greek,Primary,Finite difference,Bump,Difference,Per contract
0,delta,0.406616,0.407045,0.39420,4.296928e-04,40.661557
1,gamma,0.187513,0.186961,0.39420,-5.515410e-04,18.751274
2,vega_1pct,0.021179,0.021179,0.00100,-4.982643e-09,2.117914
3,theta_day,-0.059163,-0.061172,0.00274,-2.009341e-03,-5.916271
4,rho_1pct,0.002962,0.002962,0.00010,-1.658816e-13,0.296162


## 6. Approximation tests

In [7]:
dS=2.0; full=bs_price(S+dS,K,T,r,sig,q)-base; delta_only=G['delta']*dS; delta_gamma=delta_only+.5*G['gamma']*dS**2
dS2=1.; dsig=.02; days=1; dr=.0025; full_multi=bs_price(S+dS2,K,T-days/365,r+dr,sig+dsig,q)-base; multi=G['delta']*dS2+.5*G['gamma']*dS2**2+G['vega_1pct']*(dsig/.01)+G['theta_day']*days+G['rho_1pct']*(dr/.01)
pd.DataFrame([['Delta only',delta_only,full,delta_only-full],['Delta-gamma',delta_gamma,full,delta_gamma-full],['Multi-factor',multi,full_multi,multi-full_multi]],columns=['Method','Estimated change','Fully repriced change','Error'])

,Method,Estimated change,Fully repriced change,Error
0,Delta only,0.813231,1.182360,-0.369129
1,Delta-gamma,1.188257,1.182360,0.005897
2,Multi-factor,0.484308,0.478751,0.005557


## 7. Interactive Greek sensitivities

In [8]:
grid=np.linspace(30,50,81); rows=[]
for x in grid: rows.append({'spot':x,**bs_greeks(x,K,T,r,sig,q,M['option_type'])})
sens=pd.DataFrame(rows); fig=make_subplots(rows=2,cols=2,subplot_titles=('Delta','Gamma','Vega per 1 vol point','Daily Theta'))
for (col,rw,c) in [('delta',1,1),('gamma',1,2),('vega_1pct',2,1),('theta_day',2,2)]:
    fig.add_trace(go.Scatter(x=sens.spot,y=sens[col],name=col),row=rw,col=c)
    fig.add_vline(x=S,line_dash='dash',line_color='#1f77b4',row=rw,col=c); fig.add_vline(x=K,line_dash='dot',line_color='#d62728',row=rw,col=c)
fig.update_layout(height=700,title='YETI Nov. 20, 2026 $40 Call: Greek Sensitivities',showlegend=False); fig.show()

## 8. Integrated Decision Memo


## Equity Valuation

YETI's valuation evidence does not point to one uniquely correct share value. The prior discounted cash flow model produced $37.30 per diluted share using an approximately 11.5% WACC and 3.0% terminal growth rate, below the September 15, 2026 market price of $39.40. The comparable-company base value was $49.81, while the 60% DCF and 40% comparable blend was $42.30. The dispersion is economically reasonable: the DCF penalizes distant cash flows with a high discount rate, whereas the peer method assumes YETI deserves a premium to an outdoor and consumer-products peer group. Seven documented analyst targets ranged from $46 to $57, with a mean of $51.00 and median of $50.00. These are twelve-month opinions, not objective probabilities, and their horizon materially exceeds the selected option's seven-day life. The DCF and comparables retain a September 15 valuation date, while the option model uses November 13 market inputs, so the small stock-price difference does not eliminate timing risk.

## Option Valuation

The selected contract is the American-style YETI November 20, 2026 $40 call, quoted on November 13 with YETI at $39.42. Its bid was $0.80, ask was $1.20, and mechanical midpoint was $1.00; volume was 129 contracts and open interest was 80. The 500-step CRR value is $0.586 per share, compared with the Black-Scholes benchmark of $0.586. The analyst-scenario discounted expected payoff is $10.72, far above both. That difference is not a model failure. The analyst scenarios apply twelve-month targets to a seven-day payoff and use subjective weights, while CRR prices the option through risk-neutral replication using 37.9% implied volatility and a 4.32% rate. The $0.40 bid-ask spread is 40% of the midpoint, so liquidity and stale-quote effects are material. The option therefore trades above the selected CRR estimate under these assumptions, but the evidence does not support calling it overpriced.

## Greeks and Risk Profile

At the current inputs, delta is 0.4066, gamma is 0.1875, vega is $0.0212 per one-percentage-point volatility change, daily theta is $-0.0592, and rho is $0.0030 per one-percentage-point rate change. For one 100-share contract, these become approximately $40.66 of stock-equivalent exposure per $1 move, 18.75 delta units of curvature per $1, $2.12 per volatility point, $-5.92 per day, and $0.30 per rate point. Theta and gamma are the most economically important because the option is almost at the money with only seven days remaining. Gamma is concentrated near the $40 strike, meaning delta changes quickly as YETI crosses that level. Vega remains positive, but the short maturity limits its dollar effect. Rho is negligible because there is little time for discounting to matter. An illustrative delta hedge would short about 41 shares per long call, but that hedge would require frequent rebalancing because gamma changes delta as the stock moves.

## Approximation and Model Risk

For a $2 increase in YETI, the delta-only estimate changes option value by $0.813, while the delta-gamma estimate gives $1.188; full repricing gives $1.182. Gamma improves the estimate because the call's payoff is strongly curved near the strike. For the combined shock of a $1 stock increase, a two-percentage-point volatility increase, one day of decay, and a 25-basis-point rate increase, the multi-factor estimate is $0.484, versus $0.479 from full repricing. Finite-difference checks use a 1% stock bump, a 0.10-percentage-point volatility bump, one calendar day, and a one-basis-point rate bump. Small residual differences reflect discretization and higher-order terms. The approximation becomes less reliable for larger or simultaneous shocks because delta, gamma, vega, and theta themselves change. Additional risks include using expiration-level rather than strike-specific implied volatility, the wide quote spread, twelve-month analyst targets applied to a one-week contract, zero modeled dividends, and the distinction between an American contract and the European analytic benchmark. For a non-dividend-paying call, early exercise should not add meaningful value, which is supported by the close CRR and Black-Scholes results.

## Conclusion

The principal claim is that the selected call's risk is dominated by near-expiration stock-price curvature and time decay, not by interest rates. The most important number is daily theta of $-0.0592 per share, or about $-5.92 per contract, holding other inputs constant. The material uncertainty is volatility and quote quality: the model uses 37.9% expiration-level ATM volatility while the market midpoint comes from a wide spread. I would revise the conclusion if a documented strike-specific implied volatility or liquid same-day quote moved the 500-step CRR value outside the $0.80 to $1.20 observed market range. This analysis is educational and is not investment advice.


## 9. Export dashboard data

In [9]:
results={'model':M,'analyst_stats':stats,'states':states.to_dict('records'),'belief_value':belief_value,'crr':lat,'black_scholes_price':base,'greeks':G,'greek_validation':validation.to_dict('records'),'convergence':conv.to_dict('records'),'sensitivity':sens.to_dict('records'),'comparison':{'Analyst scenario':belief_value,'CRR':lat['price'],'Market bid':M['market_bid'],'Market midpoint':M['market_mid'],'Market ask':M['market_ask']}}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2,default=lambda o: o.item() if hasattr(o,'item') else str(o))); print('Exported docs/model_results.json')

Exported docs/model_results.json
